# Metodologia Design Science Research (DSR)

**Etapa de Pesquisa (Peffers et al., 2007):**
### 3. Design e Desenvolvimento (Design and Development)

**Objetivo Acadêmico:** Este notebook foca na ingestão de dados de calendário (feriados, pontes, períodos letivos). No contexto do DSR, isso representa a inclusão de variáveis de contexto externas que explicam a variabilidade estrutural da demanda, essencial para a construção de um artefato resiliente e adaptável ao ambiente institucional.


# 01d - Ingestão de Calendários Multi-Ano (PDF)
Este notebook unifica calendários de diferentes anos e formatos em uma série temporal contínua.

In [5]:
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import os
import glob
import re
import unicodedata
from pypdfium2 import PdfDocument

# 1. Configurações
DIRETORIO_PDFS = "../data/calendarios/"
SAIDA_CSV = "../data/calendario_consolidado.csv"

def normalizar_texto(s, manter_espacos=True):
    if not isinstance(s, str): return ""
    # Remove acentos e simplifica caracteres
    n = unicodedata.normalize('NFD', s).encode('ascii', 'ignore').decode('utf-8')
    if manter_espacos:
        n = re.sub(r'[^a-zA-Z0-9\s]', ' ', n)
    else:
        n = re.sub(r'[^a-zA-Z0-9]', '', n)
    return n.upper().strip()

def extrair_texto_pdf(caminho):
    try:
        pdf = PdfDocument(caminho)
        return "\n".join([page.get_textpage().get_text_bounded() for page in pdf])
    except: return ""

def safe_timestamp(y, m, d):
    try:
        return pd.Timestamp(year=y, month=m, day=d)
    except ValueError:
        return None

def parse_eventos_universal(texto, ano_ref):
    eventos = []
    totais_pdf = []
    
    # 1. Pré-processamento: unificar linhas e tratar hífens colados (ex: 31-INICIO -> 31 - INICIO)
    # Regex: dígito colado em hífen seguido de letra
    texto_norm = re.sub(r'(\d)([-–:])([A-Za-z])', r'\1 \2 \3', texto)
    
    # 2. Separar por Semestres
    partes_semestre = re.split(r'(1.? SEMESTRE|2.? SEMESTRE)', texto_norm.upper())
    if len(partes_semestre) == 1:
        partes_semestre = ['', '1 SEMESTRE', texto_norm.upper()]

    for i in range(1, len(partes_semestre), 2):
        semestre_str = partes_semestre[i]
        conteudo = partes_semestre[i+1]
        mes_offset = 1 if '1' in semestre_str else 7
        
        conteudo_limpo = re.split(r'LEGENDA|DIAS LETIVOS PRAZOS DIVERSOS', conteudo)[0]
        blocos_tabela = re.split(r'DOM\..*?SAB\.', conteudo_limpo, flags=re.DOTALL)
        
        mes_atual = mes_offset
        for j in range(1, len(blocos_tabela)):
            if mes_atual > (mes_offset + 5): break 
            bloco = blocos_tabela[j]
            total_match = re.search(r"DIAS LETIVOS NO MES\s*(\d+)", bloco)
            if total_match:
                totais_pdf.append({'ano': ano_ref, 'mes': mes_atual, 'total_pdf': int(total_match.group(1))})
            
            # Capturar eventos: suporta mutiplas linhas e hífens variados
            blocos_eventos = re.split(r'\n(?=\d{1,2}(?:\s*(?:A|E|/)\s*\d{1,2})?\s*[-–:])', bloco.strip())
            for be in blocos_eventos:
                m = re.search(r'^(\d{1,2}(?:\s*(?:A|E|/)\s*\d{1,2})?)\s*[-–:]\s*(.+)', be.strip(), re.DOTALL)
                if m:
                    dias_str, desc = m.group(1), m.group(2).replace('\n', ' ').strip()
                    if desc.upper() in ['JANEIRO', 'FEVEREIRO', 'MARCO', 'ABRIL', 'MAIO', 'JUNHO', 'JULHO', 'AGOSTO', 'SETEMBRO', 'OUTUBRO', 'NOVEMBRO', 'DEZEMBRO']: continue
                    
                    nums = re.findall(r'\d+', dias_str)
                    if ' A ' in dias_str.upper() and len(nums) == 2:
                        for d in range(int(nums[0]), int(nums[1]) + 1):
                            ts = safe_timestamp(ano_ref, mes_atual, d)
                            if ts: eventos.append({'data': ts, 'evento': desc})
                    else:
                        for d_str in nums:
                            ts = safe_timestamp(ano_ref, mes_atual, int(d_str))
                            if ts: eventos.append({'data': ts, 'evento': desc})
            mes_atual += 1
            
    # 3. Captura GLOBAL de Intervalos (ex: 31/05 A 03/06)
    pattern_intervalo_mes = re.finditer(r'(\d{1,2})/(\d{1,2})\s*A\s*(\d{1,2})/(\d{1,2})\s*[-–:]\s*([^-–:\n]+)', texto_norm.upper())
    for m in pattern_intervalo_mes:
        d1, m1, d2, m2, desc = m.groups()
        ts_ini = safe_timestamp(ano_ref, int(m1), int(d1))
        ts_fim = safe_timestamp(ano_ref, int(m2), int(d2))
        if ts_ini and ts_fim:
            for dt in pd.date_range(ts_ini, ts_fim):
                eventos.append({'data': dt, 'evento': desc.strip()})

    # 4. Captura Global de Marcos DD/MM
    marcos = re.finditer(r'(\d{1,2})/(\d{1,2})\s*[-–:]\s*([^-–:\n]{3,})', texto_norm.upper())
    for m in marcos:
        ts = safe_timestamp(ano_ref, int(m.group(2)), int(m.group(1)))
        if ts: eventos.append({'data': ts, 'evento': m.group(3).strip()})

    return pd.DataFrame(eventos), pd.DataFrame(totais_pdf)

# Execução Multi-Ano
arquivos = glob.glob(os.path.join(DIRETORIO_PDFS, "*.pdf"))
todos, todos_totais = [], []
anos = set()

for f in arquivos:
    txt = extrair_texto_pdf(f)
    ano_match = re.search(r'20\d{2}', os.path.basename(f) + txt)
    ano_ref = int(ano_match.group()) if ano_match else 2025
    print(f"📖 Processando: {os.path.basename(f)} (Ano: {ano_ref})")
    
    df_pdf, df_totais = parse_eventos_universal(txt, ano_ref)
    
    if not df_pdf.empty:
        print(f"   Eventos extraídos: {len(df_pdf)}")
        print(f"   Amostra: {df_pdf['evento'].unique()[:3]}") # Ver nomes reais
        todos.append(df_pdf)
        anos.update(df_pdf['data'].dt.year.unique())
    else:
        print("   ⚠️ Nenhum evento extraído deste PDF.")
        
    if not df_totais.empty:
        todos_totais.append(df_totais)

if todos:
    # Consolidação: UNIR descrições na mesma data para não perder marcos importantes
    df_unificado = pd.concat(todos).drop_duplicates(subset=['data', 'evento'])
    df_unificado = df_unificado.groupby('data')['evento'].apply(lambda x: ' / '.join(x.unique())).reset_index()
    
    ano_min, ano_max = min(anos), max(anos)
    datas_gap = pd.date_range(start=f"{ano_min}-01-01", end=f"{ano_max}-12-31")
    df_final = pd.DataFrame({'data': datas_gap})
    df_final['dia_semana_num'] = df_final['data'].dt.dayofweek
    
    df_final['evento_padrao'] = 'DIA LETIVO PADRAO'
    df_final.loc[df_final['dia_semana_num'] == 5, 'evento_padrao'] = 'SABADO'
    df_final.loc[df_final['dia_semana_num'] == 6, 'evento_padrao'] = 'DOMINGO'
    
    df_final = pd.merge(df_final, df_unificado, on='data', how='left')
    df_final['evento'] = df_final['evento'].fillna(df_final['evento_padrao'])
    df_final['norm_evento'] = df_final['evento'].apply(normalizar_texto)
    
    # 1. Feriados e Pontos Facultativos
    df_final['eh_feriado'] = df_final['norm_evento'].str.contains("FERIADO|RECESSO|CARNAVAL|CONFRATERNIZACAO|MUNDIAL DA PAZ|DIA DO TRABALHO|INDEPENDENCIA|FINADOS|PROCLAMACAO|NATAL|PONTO FACULTATIVO")
    df_final['vespera_feriado'] = df_final['eh_feriado'].shift(-1).fillna(False)
    
    # 2. Reuniões de Impacto (Expandido)
    # Inclui variações de Reunião, Conselho e NAPNE
    regex_reuniao = r"(?:PAIS|FAMILIA|FAMILIAS|REPLANEJAMENTO|CONSELHO|REAVALIACAO|NAPNE)"
    df_final['eh_reuniao_impacto'] = df_final['norm_evento'].str.contains(regex_reuniao, regex=True)
    
    # 3. Eventos Especiais (Semanas e Festas)
    # Captura Semanas, Festas Juninas, Portas Abertas, SNCT, etc.
    regex_especial = r"(?:SEMANA|FESTA JUNINA|CAMPUS PORTAS ABERTAS|SNCT|CONSCIENCIA NEGRA|MOSTRA|EXPOSICAO|CULTURA|DIVERSIDADE)"
    df_final['eh_evento_especial'] = df_final['norm_evento'].str.contains(regex_especial, regex=True)
    
    # 4. Detectar Etapas Acadêmicas (Com Estimativa Robusta)
    df_final['etapa'] = 0
    etapas_detectadas = []
    
    for ano in df_final['data'].dt.year.unique():
        mask_ano = (df_final['data'].dt.year == ano)
        df_ano = df_final[mask_ano]
        
        for i in range(1, 5):
            # Regex mais permissiva para capturar '31- INICIO' ou '03-INICIO' ou 'BIMESTRE'
            pat_ini = rf"(?:INICIO|COMECO).*?(?:ETAPA {i}|{i}.? ETAPA|{i}.? BIMESTRE)"
            pat_fim = rf"(?:ENCERRAMENTO|TERMINO|FINAL).*?(?:ETAPA {i}|{i}.? ETAPA|{i}.? BIMESTRE)"
            
            mask_ini = df_ano['norm_evento'].str.contains(pat_ini, regex=True, case=False)
            mask_fim = df_ano['norm_evento'].str.contains(pat_fim, regex=True, case=False)
            
            ini_idx = df_ano.loc[mask_ini].index.min() if mask_ini.any() else None
            fim_idx = df_ano.loc[mask_fim].index.max() if mask_fim.any() else None
            
            tipo = "Detectado"
            if ini_idx is None or fim_idx is None:
                tipo = "Estimado"
                if i == 1: d_ini, d_fim = f"{ano}-02-05", f"{ano}-04-20"
                elif i == 2: d_ini, d_fim = f"{ano}-04-22", f"{ano}-07-08"
                elif i == 3: d_ini, d_fim = f"{ano}-07-25", f"{ano}-10-05"
                else: d_ini, d_fim = f"{ano}-10-06", f"{ano}-12-15"
                
                try:
                    ts_ini, ts_fim = pd.Timestamp(d_ini), pd.Timestamp(d_fim)
                    # Se tivermos uma data sugerida no CSV próximo ao marco estimado, usar ela
                    ini_idx = df_final[df_final['data'] == ts_ini].index.min()
                    fim_idx = df_final[df_final['data'] == ts_fim].index.max()
                except: pass

            if ini_idx is not None and fim_idx is not None:
                df_final.loc[ini_idx:fim_idx, 'etapa'] = i
                etapas_detectadas.append({
                    'Ano': ano, 'Etapa': f'Etapa {i}', 'Tipo': tipo,
                    'Inicio': df_final.loc[ini_idx, 'data'].strftime('%d/%m/%Y'), 
                    'Fim': df_final.loc[fim_idx, 'data'].strftime('%d/%m/%Y')
                })
    
    df_etapas_obs = pd.DataFrame(etapas_detectadas)
    
    # 4. Regra de Negócio: tem_refeicao (Ajustado para Multi-Ano)
    # Regex mais permissiva para capturar variações de 'INICIO' e marcos de aula
    pat_inicio_voto = r"(?:INICIO|COMECO|RETORNO).*?(?:AULAS|ETAPA 1|1.? ETAPA|LETIVO)"
    mask_inicio_aulas = df_final['norm_evento'].str.contains(pat_inicio_voto, regex=True)
    df_final['tem_refeicao'] = True
    
    # Identificar início das aulas por ano
    for ano in df_final['data'].dt.year.unique():
        mask_ano = (df_final['data'].dt.year == ano)
        inicio_ano = df_final.loc[mask_ano & mask_inicio_aulas, 'data'].min()
        if pd.notna(inicio_ano):
            df_final.loc[mask_ano & (df_final['data'] < inicio_ano), 'tem_refeicao'] = False
            print(f"✅ Ano {ano}: Marcou início de refeições em {inicio_ano.strftime('%d/%m/%Y')}")
        else:
            # Fallback seguro: se não achar marco, assumir que começa no primeiro dia útil de Fevereiro
            fallback = pd.Timestamp(year=ano, month=2, day=1)
            while fallback.dayofweek >= 5: fallback += pd.Timedelta(days=1)
            df_final.loc[mask_ano & (df_final['data'] < fallback), 'tem_refeicao'] = False
            print(f"⚠️ Ano {ano}: Marco de INICIO DAS AULAS não encontrado. Usando fallback {fallback.strftime('%d/%m/%Y')}")

    df_final.loc[df_final['eh_feriado'] | (df_final['dia_semana_num'] >= 5), 'tem_refeicao'] = False
    
    # Classificação Concluída
    
    
    # 6. Preparar Exportação
    df_export = df_final[['data', 'evento', 'eh_feriado', 'vespera_feriado', 'eh_reuniao_impacto', 'tem_refeicao', 'eh_evento_especial', 'etapa']].copy()
    bool_cols = ['eh_feriado', 'vespera_feriado', 'eh_reuniao_impacto', 'tem_refeicao', 'eh_evento_especial']
    df_export[bool_cols] = df_export[bool_cols].astype(int)
    df_export.to_csv(SAIDA_CSV, index=False)
    
    # AUDITORIA AUTOMÁTICA
    if todos_totais:
        df_totais_pdf = pd.concat(todos_totais).drop_duplicates()
        df_calc = df_final.groupby([df_final['data'].dt.year.rename('ano'), df_final['data'].dt.month.rename('mes')])['tem_refeicao'].sum().reset_index()
        df_calc.columns = ['ano', 'mes', 'total_calc']
        df_conf = pd.merge(df_totais_pdf, df_calc, on=['ano', 'mes'], how='inner')
        df_conf['status'] = df_conf.apply(lambda x: '✅ OK' if x['total_pdf'] == x['total_calc'] else f'❌ Erro ({x["total_calc"]-x["total_pdf"]}d)', axis=1)
        df_conf.columns = ['Ano', 'Mês', 'Total PDF', 'Total Calculado', 'Status']
    
    print(f"🚀 SUCESSO! Base multi-ano refinada em: {SAIDA_CSV}")
else:
    print("⚠️ Falha crítica: Nenhum dado extraído.")

📖 Processando: CALENDARIO_2023.pdf (Ano: 2023)
   Eventos extraídos: 22
   Amostra: ['PRIMEIRO SEMESTRE\r 1 09 - REVOLUÇÃO /MOVIMENTO CONSTITUCIONALISTA DE 1932 - LEI ESTADUAL Nº 9.497, DE 05/03/1997 (FERIADO ESTADUAL)\r 2 3 4 5 6 7 8 10 A 24 - FÉRIAS DOCENTES E DISCENTES (15 DIAS CORRIDOS) - PORTARIA Nº 2.791, DE 08 DE DEZEMBRO DE 2010\r 9 10 11 12 13 14 15 25 E 26 - RETORNO ÀS ATIVIDADES / PLANEJAMENTO PEDAGÓGICO E REUNIÕES ADMINISTRATIVAS\r 16 17 18 19 20 21 22 27 - INÍCIO DO 3º BIMESTRE\r 23 24 25 26 27 28 29\r 30 31\r DIAS 1 0 0 1 1 0\r ACUMULADO 1 0 0 1 1 0\r DIAS LETIVOS NO MÊS 3\r DIAS LETIVOS ACUMULADOS NO SEMESTRE 3\r DIAS LETIVOS ACUMULADOS NO ANO 107\r ATIVIDADES / EVENTOS'
 'REUNIÃO COM AS FAMÍLIAS\r 1 2 3 4 5 18 - PRAZO MÁXIMO PARA CANCELAMENTO DE MATRÍCULA DAS DISCIPLINAS OPTATIVAS NO 2º SEM. (LIBRAS E ESPANHOL)\r 6 7 8 9 10 11 12 26 - SÁBADO LETIVO - ATIVIDADES PEDAGÓGICAS PROGRAMADAS/ REPOSIÇÃO DE AULAS DE 6ª FEIRA\r 13 14 15 16 17 18 19 30 - REUNIÃO COM A EQUIPE DE FO

# 7. Sumário da Ingestão de Calendário
Resumo quantitativo dos dias letivos, feriados e eventos extraídos.

In [6]:
# Sumário de Ingestão e Auditoria de Qualidade
import pandas as pd
df_temp = df_export.copy()
df_temp['data'] = pd.to_datetime(df_temp['data'])
total_dias = len(df_temp)
dias_letivos = df_temp['tem_refeicao'].sum()
total_feriados = df_temp['eh_feriado'].sum()
total_reunioes = df_temp['eh_reuniao_impacto'].sum()
eventos_especiais = df_temp['eh_evento_especial'].sum()
primeira_data = df_temp['data'].min().strftime('%d/%m/%Y')
ultima_data = df_temp['data'].max().strftime('%d/%m/%Y')

print(f"--- RELATÓRIO DE INGESTÃO (CALENDÁRIO REFINADO) ---")
print(f"Total de Dias no Período: {total_dias}")
print(f"Total de Dias Letivos (Aulas Pós-Início): {dias_letivos}")
print(f"Total de Feriados/Pontos Facultativos: {total_feriados}")
print(f"Total de Reuniões de Impacto: {total_reunioes}")
print(f"Total de Eventos Especiais: {eventos_especiais}")
print(f"Período: {primeira_data} até {ultima_data}")

# 1. Cronograma de Etapas Acadêmicas
print("\nCronograma de Etapas Acadêmicas:")
if 'df_etapas_obs' in locals() and not df_etapas_obs.empty:
    print(df_etapas_obs.to_string(index=False))
else:
    print("Nenhuma etapa detectada explicitamente.")

# 2. Auditoria de Integridade (PDF vs CSV)
print("\nAuditoria de Integridade (Dias Letivos por Mês):")
if 'df_conf' in locals() and not df_conf.empty:
    display(df_conf)
else:
    print("Informação de totais não disponível no PDF para comparação automática.")

# 3. Visão Detalhada para Conferência Manual
print("\nDetalhamento para Conferência Manual (Dias com Alterações):")
mask_conferencia = (df_temp['eh_feriado']==1) | \
                    (df_temp['eh_reuniao_impacto']==1) | \
                    (df_temp['eh_evento_especial']==1) | \
                    (df_temp['etapa'] > 0) | \
                    (~df_temp['evento'].isin(['DIA LETIVO PADRAO', 'SABADO', 'DOMINGO']))

detalhe_cal = df_temp[mask_conferencia].copy()
detalhe_cal = detalhe_cal[['data', 'etapa', 'evento', 'tem_refeicao', 'eh_feriado', 'eh_reuniao_impacto', 'eh_evento_especial']]
detalhe_cal.columns = ['Data', 'Etapa', 'Evento/Status', 'Letivo', 'Feriado', 'Reunião', 'Evento Esp.']
detalhe_cal['Data'] = detalhe_cal['Data'].dt.strftime('%d/%m/%Y')

display(detalhe_cal.style.apply(lambda x: ['background-color: #e6fffa' if v == 1 else '' for v in x], subset=['Letivo'])
                  .apply(lambda x: ['background-color: #fff5f5' if v == 1 else '' for v in x], subset=['Feriado'])
                  .apply(lambda x: ['background-color: #fffaf0' if v == 1 else '' for v in x], subset=['Reunião'])
                  .apply(lambda x: ['background-color: #e6f4ff' if v == 1 else '' for v in x], subset=['Evento Esp.'])
                  .apply(lambda x: ['font-weight: bold; color: #2c5282' if v > 0 else '' for v in x], subset=['Etapa']))

--- RELATÓRIO DE INGESTÃO (CALENDÁRIO REFINADO) ---
Total de Dias no Período: 1096
Total de Dias Letivos (Aulas Pós-Início): 701
Total de Feriados/Pontos Facultativos: 42
Total de Reuniões de Impacto: 32
Total de Eventos Especiais: 14
Período: 01/01/2023 até 31/12/2025

Cronograma de Etapas Acadêmicas:
 Ano   Etapa      Tipo     Inicio        Fim
2023 Etapa 1  Estimado 05/02/2023 20/04/2023
2023 Etapa 2  Estimado 22/04/2023 08/07/2023
2023 Etapa 3 Detectado 03/01/2023 04/04/2023
2023 Etapa 4  Estimado 06/10/2023 15/12/2023
2024 Etapa 1  Estimado 05/02/2024 20/04/2024
2024 Etapa 2  Estimado 22/04/2024 08/07/2024
2024 Etapa 3  Estimado 25/07/2024 05/10/2024
2024 Etapa 4  Estimado 06/10/2024 15/12/2024
2025 Etapa 1 Detectado 03/02/2025 11/04/2025
2025 Etapa 2  Estimado 22/04/2025 08/07/2025
2025 Etapa 3 Detectado 09/07/2025 02/10/2025
2025 Etapa 4 Detectado 02/10/2025 06/12/2025

Auditoria de Integridade (Dias Letivos por Mês):
Informação de totais não disponível no PDF para comparação au

,Data,Etapa,Evento/Status,Letivo,Feriado,Reunião,Evento Esp.
2,03/01/2023,3,"PRIMEIRO SEMESTRE 1 09 - REVOLUÇÃO /MOVIMENTO CONSTITUCIONALISTA DE 1932 - LEI ESTADUAL Nº 9.497, DE 05/03/1997 (FERIADO ESTADUAL) 2 3 4 5 6 7 8 10 A 24 - FÉRIAS DOCENTES E DISCENTES (15 DIAS CORRIDOS) - PORTARIA Nº 2.791, DE 08 DE DEZEMBRO DE 2010 9 10 11 12 13 14 15 25 E 26 - RETORNO ÀS ATIVIDADES / PLANEJAMENTO PEDAGÓGICO E REUNIÕES ADMINISTRATIVAS 16 17 18 19 20 21 22 27 - INÍCIO DO 3º BIMESTRE 23 24 25 26 27 28 29 30 31 DIAS 1 0 0 1 1 0 ACUMULADO 1 0 0 1 1 0 DIAS LETIVOS NO MÊS 3 DIAS LETIVOS ACUMULADOS NO SEMESTRE 3 DIAS LETIVOS ACUMULADOS NO ANO 107 ATIVIDADES / EVENTOS",0,1,0,0
3,04/01/2023,3,"PRIMEIRO SEMESTRE 1 09 - REVOLUÇÃO /MOVIMENTO CONSTITUCIONALISTA DE 1932 - LEI ESTADUAL Nº 9.497, DE 05/03/1997 (FERIADO ESTADUAL) 2 3 4 5 6 7 8 10 A 24 - FÉRIAS DOCENTES E DISCENTES (15 DIAS CORRIDOS) - PORTARIA Nº 2.791, DE 08 DE DEZEMBRO DE 2010 9 10 11 12 13 14 15 25 E 26 - RETORNO ÀS ATIVIDADES / PLANEJAMENTO PEDAGÓGICO E REUNIÕES ADMINISTRATIVAS 16 17 18 19 20 21 22 27 - INÍCIO DO 3º BIMESTRE 23 24 25 26 27 28 29 30 31 DIAS 1 0 0 1 1 0 ACUMULADO 1 0 0 1 1 0 DIAS LETIVOS NO MÊS 3 DIAS LETIVOS ACUMULADOS NO SEMESTRE 3 DIAS LETIVOS ACUMULADOS NO ANO 107 ATIVIDADES / EVENTOS",0,1,0,0
4,05/01/2023,3,"PRIMEIRO SEMESTRE 1 09 - REVOLUÇÃO /MOVIMENTO CONSTITUCIONALISTA DE 1932 - LEI ESTADUAL Nº 9.497, DE 05/03/1997 (FERIADO ESTADUAL) 2 3 4 5 6 7 8 10 A 24 - FÉRIAS DOCENTES E DISCENTES (15 DIAS CORRIDOS) - PORTARIA Nº 2.791, DE 08 DE DEZEMBRO DE 2010 9 10 11 12 13 14 15 25 E 26 - RETORNO ÀS ATIVIDADES / PLANEJAMENTO PEDAGÓGICO E REUNIÕES ADMINISTRATIVAS 16 17 18 19 20 21 22 27 - INÍCIO DO 3º BIMESTRE 23 24 25 26 27 28 29 30 31 DIAS 1 0 0 1 1 0 ACUMULADO 1 0 0 1 1 0 DIAS LETIVOS NO MÊS 3 DIAS LETIVOS ACUMULADOS NO SEMESTRE 3 DIAS LETIVOS ACUMULADOS NO ANO 107 ATIVIDADES / EVENTOS",0,1,0,0
5,06/01/2023,3,"PRIMEIRO SEMESTRE 1 09 - REVOLUÇÃO /MOVIMENTO CONSTITUCIONALISTA DE 1932 - LEI ESTADUAL Nº 9.497, DE 05/03/1997 (FERIADO ESTADUAL) 2 3 4 5 6 7 8 10 A 24 - FÉRIAS DOCENTES E DISCENTES (15 DIAS CORRIDOS) - PORTARIA Nº 2.791, DE 08 DE DEZEMBRO DE 2010 9 10 11 12 13 14 15 25 E 26 - RETORNO ÀS ATIVIDADES / PLANEJAMENTO PEDAGÓGICO E REUNIÕES ADMINISTRATIVAS 16 17 18 19 20 21 22 27 - INÍCIO DO 3º BIMESTRE 23 24 25 26 27 28 29 30 31 DIAS 1 0 0 1 1 0 ACUMULADO 1 0 0 1 1 0 DIAS LETIVOS NO MÊS 3 DIAS LETIVOS ACUMULADOS NO SEMESTRE 3 DIAS LETIVOS ACUMULADOS NO ANO 107 ATIVIDADES / EVENTOS",0,1,0,0
6,07/01/2023,3,"PRIMEIRO SEMESTRE 1 09 - REVOLUÇÃO /MOVIMENTO CONSTITUCIONALISTA DE 1932 - LEI ESTADUAL Nº 9.497, DE 05/03/1997 (FERIADO ESTADUAL) 2 3 4 5 6 7 8 10 A 24 - FÉRIAS DOCENTES E DISCENTES (15 DIAS CORRIDOS) - PORTARIA Nº 2.791, DE 08 DE DEZEMBRO DE 2010 9 10 11 12 13 14 15 25 E 26 - RETORNO ÀS ATIVIDADES / PLANEJAMENTO PEDAGÓGICO E REUNIÕES ADMINISTRATIVAS 16 17 18 19 20 21 22 27 - INÍCIO DO 3º BIMESTRE 23 24 25 26 27 28 29 30 31 DIAS 1 0 0 1 1 0 ACUMULADO 1 0 0 1 1 0 DIAS LETIVOS NO MÊS 3 DIAS LETIVOS ACUMULADOS NO SEMESTRE 3 DIAS LETIVOS ACUMULADOS NO ANO 107 ATIVIDADES / EVENTOS",0,1,0,0
7,08/01/2023,3,DOMINGO,0,0,0,0
8,09/01/2023,3,DIA LETIVO PADRAO,1,0,0,0
9,10/01/2023,3,DIA LETIVO PADRAO,1,0,0,0
10,11/01/2023,3,DIA LETIVO PADRAO,1,0,0,0
11,12/01/2023,3,DIA LETIVO PADRAO,1,0,0,0
